# RSNA Knee Abnormality Detection — Frozen Image Baseline

Builds the Phase 3B image baseline exactly as specified: one validated series per anatomical plane, five central-band slices each, physical-aspect letterboxing, DICOM-faithful intensity normalization, signed laterality canonicalization, a frozen DINOv2-small encoder, shared-mean aggregation to one 388-dimensional study vector, and a single strongly regularized multilabel linear head evaluated out-of-fold on the 58 human-labeled studies.

Every configuration value below is frozen before this notebook runs and is displayed rather than described, so the exact contract is visible in the output. Every result is an aggregate count, rate, or score — no report text, no study or series identifiers, no per-study predictions, and no pixel data is displayed or persisted.

In [ ]:
import hashlib
import importlib.metadata
import importlib.util
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError("This notebook runs on Kaggle only.")

# Verify and install the pinned offline wheel BEFORE inserting the source
# path or importing knee_mri anywhere, so the import cannot silently pick up
# a different stratifier than the one this contract pins.
package_initializers = tuple(Path("/kaggle/input/datasets").rglob("knee_mri/__init__.py"))
if len(package_initializers) != 1:
    raise RuntimeError("Expected exactly one attached knee_mri source package.")
_src_root = package_initializers[0].parent.parent
_dataset_root = _src_root.parent

WHEEL_NAME = "iterative_stratification-0.1.9-py3-none-any.whl"
EXPECTED_SHA256 = "476f8deff6753fb1725612fe41e59cc2058f8f2524ae5d1ccee88eb8c8d3de80"

wheel_matches = tuple(_dataset_root.rglob(WHEEL_NAME))
if len(wheel_matches) != 1:
    raise RuntimeError("Expected exactly one pinned iterative-stratification wheel.")
wheel_path = wheel_matches[0]
if hashlib.sha256(wheel_path.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("Pinned iterative-stratification wheel checksum mismatch.")

try:
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", str(wheel_path)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
except OSError:
    raise RuntimeError("Failed to launch the offline install for the pinned wheel.") from None
if install_result.returncode != 0:
    raise RuntimeError("Offline installation of the pinned wheel failed.")
if importlib.metadata.version("iterative-stratification") != "0.1.9":
    raise RuntimeError("Installed iterative-stratification version mismatch.")

# The processor statistics come from the vendored copy of the attached
# model's own preprocessor_config.json. There is deliberately no fallback:
# substituting remembered constants is the silent-wrongness this contract
# exists to prevent.
PROCESSOR_CONFIG_NAME = "dinov2-small-preprocessor_config.json"
processor_matches = tuple(_dataset_root.rglob(PROCESSOR_CONFIG_NAME))
if len(processor_matches) != 1:
    raise RuntimeError("Expected exactly one vendored DINOv2 processor config.")
PROCESSOR_CONFIG_PATH = processor_matches[0]

# Install the vendored DICOM codec plugins. The corpus-wide census found no
# compressed series in either released split, so these are insurance for the
# hidden set rather than a current requirement -- but an undecodable slice
# there would fail silently into the fallback row, which is the failure mode
# worth spending a few seconds to avoid.
#
# --no-deps is required, not stylistic: both compiled wheels declare
# numpy>=2.0,<3.0, and without it pip would try to resolve or replace the
# kernel's own numpy, offline and unasked.
CODEC_WHEELS = {
    "pylibjpeg-2.1.0-py3-none-any.whl":
        "25df9496a69e64e98c887fddee12a1271e275b5f74ba804f9bf98a08bb80993e",
    "pylibjpeg_openjpeg-2.5.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl":
        "a22fcb649ba9849209d8e43dba88632445a5941f0cd6765338b3652a4c686140",
    "pylibjpeg_libjpeg-2.4.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl":
        "01d950ef496476a9223e4966376cb88098fcf5c55a12a21f722d7b5f84daae43",
}

codec_paths = []
for codec_name, codec_sha256 in CODEC_WHEELS.items():
    matches = tuple(_dataset_root.rglob(codec_name))
    if len(matches) != 1:
        raise RuntimeError("Expected exactly one copy of each vendored codec wheel.")
    if hashlib.sha256(matches[0].read_bytes()).hexdigest() != codec_sha256:
        raise RuntimeError("Vendored codec wheel checksum mismatch.")
    codec_paths.append(str(matches[0]))

try:
    codec_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", *codec_paths],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
except OSError:
    raise RuntimeError("Failed to launch the offline install for the codec wheels.") from None
if codec_result.returncode != 0:
    raise RuntimeError("Offline installation of the codec wheels failed.")

sys.path.insert(0, str(_src_root))

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

In [ ]:
import torch
from transformers import AutoModel

from knee_mri.dataset import split_labeled_studies, validated_plane_candidates
from knee_mri.image_model import (
    CONTINUOUS_DIMENSIONS,
    IMAGE_CLASSIFIER_C,
    build_image_classifier,
    cross_validate_image_model,
    fit_image_model,
    fold_signature,
)
from knee_mri.intensity import load_processor_statistics
from knee_mri.labels import LABEL_COLUMNS
from knee_mri.laterality import DOMINANCE_GATE, SeriesLateralityEvidence, study_laterality
from knee_mri.metrics import macro_auc
from knee_mri.model_selection import select_multilabel_folds
from knee_mri.series_audit import audit_series
from knee_mri.slice_sampling import (
    MINIMUM_DECODED_SLICES,
    SLICE_SAMPLE_SIZE,
    select_plane_sample,
)
from knee_mri.study_features import (
    EMBEDDING_DIM,
    PLANES,
    STUDY_VECTOR_DIM,
    PlaneInput,
    build_study_features,
)
from knee_mri.submission import build_submission

## 1. Frozen Configuration

In [ ]:
IMAGE_MEAN, IMAGE_STD = load_processor_statistics(PROCESSOR_CONFIG_PATH)

frozen_contract = pd.Series(
    {
        "Study vector dimensions": STUDY_VECTOR_DIM,
        "Embedding dimensions": EMBEDDING_DIM,
        "Presence + reliability flags": STUDY_VECTOR_DIM - EMBEDDING_DIM,
        "Slices sampled per plane": SLICE_SAMPLE_SIZE,
        "Minimum decoded slices per plane": MINIMUM_DECODED_SLICES,
        "Laterality dominance gate": DOMINANCE_GATE,
        "Classifier C": IMAGE_CLASSIFIER_C,
        "Classifier penalty": build_image_classifier().estimator.penalty,
        "Classifier solver": build_image_classifier().estimator.solver,
        "Classifier class_weight": build_image_classifier().estimator.class_weight,
        "Classifier max_iter": build_image_classifier().estimator.max_iter,
        "Fold candidates": "(5, 4, 3, 2)",
        "Fold seed": SEED,
        "Scaled dimensions (flags unscaled)": CONTINUOUS_DIMENSIONS,
        "Processor image_mean": str(IMAGE_MEAN),
        "Processor image_std": str(IMAGE_STD),
    },
    name="Value",
).to_frame()

display(frozen_contract)

**Interpretation:** every value the pipeline depends on, read back from its frozen source rather than restated by hand, so a silent divergence between what this run did and what it was meant to do is visible here rather than inferred later from a score. `Laterality dominance gate` is a cost-asymmetry safety choice supported by measured coverage, not an empirical separation point — accepting a badly oblique acquisition would corrupt an input invisibly, while rejecting a usable one merely leaves it untransformed and flagged. `Classifier C` is deliberately `0.1` rather than the report model's `1.0`: that value suited 50,000 sparse text features, whereas this is roughly 388 dense features on the same 58 studies, a far higher per-feature overfitting risk. `Processor image_mean`/`image_std` are read from the attached model's own configuration; there is no remembered-constant fallback anywhere in this pipeline, because a plausible-but-wrong normalization is precisely the kind of error nothing downstream would catch.

## 2. Frozen Encoder and Environment

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Expected a GPU-enabled kernel for the image baseline.")


def _find_dinov2_dir(root: Path) -> Path:
    for config_path in root.rglob("config.json"):
        try:
            config = json.loads(config_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if config.get("model_type") == "dinov2":
            return config_path.parent
    raise RuntimeError("Could not find an attached DINOv2 model source.")


DEVICE = torch.device("cuda")
_dinov2_dir = _find_dinov2_dir(Path("/kaggle/input"))
dinov2 = AutoModel.from_pretrained(str(_dinov2_dir), local_files_only=True)
dinov2 = dinov2.to(DEVICE).eval()
for parameter in dinov2.parameters():
    parameter.requires_grad_(False)

cuda_major, cuda_minor = torch.cuda.get_device_capability(0)
GPU_COMPATIBLE = f"sm_{cuda_major}{cuda_minor}" in torch.cuda.get_arch_list()
if not GPU_COMPATIBLE:
    raise RuntimeError("Allocated GPU compute capability unsupported by installed PyTorch.")


def encode_batch(batch: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        outputs = dinov2(pixel_values=batch.to(DEVICE), interpolate_pos_encoding=True)
    return outputs.last_hidden_state[:, 0, :].detach().cpu()


# Smoke test: a checksum proves the bytes, not that the plugin loads.
codec_plugins = {
    name: importlib.util.find_spec(name) is not None
    for name in ("pylibjpeg", "libjpeg", "openjpeg")
}
if not all(codec_plugins.values()):
    raise RuntimeError("A vendored codec plugin failed to import after install.")

environment_summary = pd.Series(
    {
        "torch version": importlib.metadata.version("torch"),
        "transformers version": importlib.metadata.version("transformers"),
        "CUDA device": torch.cuda.get_device_name(0),
        "CUDA compute capability": f"{cuda_major}.{cuda_minor}",
        "DINOv2 parameters": sum(p.numel() for p in dinov2.parameters()),
        "Codec plugins importable": str(sorted(codec_plugins)),
        "Encoder trainable parameters": sum(
            p.numel() for p in dinov2.parameters() if p.requires_grad
        ),
    },
    name="Value",
).to_frame()

display(environment_summary)

**Interpretation:** records the exact runtime these scores were produced on, since a timing or numerical result is only meaningful alongside the hardware and library versions behind it. `Encoder trainable parameters` must be `0`: the encoder is frozen by design, and one that quietly trained would surface only as an unexplained score. An incompatible GPU raises an error rather than being skipped, because a partial run would still produce a submission — from an untested path.

## 3. Study Feature Extraction

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_series_df = pd.read_csv(DATA_DIR / "train_series.csv")
test_series_df = pd.read_csv(DATA_DIR / "test_series.csv")

labeled_studies, _ = split_labeled_studies(train_df)
labeled_studies = labeled_studies.reset_index(drop=True)

# Aggregate-only telemetry. Every entry is a count or a rate; no study or
# series identifier is ever placed in these structures.
telemetry = {
    "planes_absent": 0,
    "plane_retries": 0,
    "candidates_tried": 0,
    "decoded_slice_counts": [],
    "laterality_unreliable_studies": 0,
    "studies_with_no_plane": 0,
    "header_read_failures": 0,
}


def _study_laterality(series_df: pd.DataFrame, series_root: Path, study_id: str):
    """Conservative consensus over EVERY available series in the study."""
    evidence = []
    study_dir = series_root / study_id
    if not study_dir.is_dir():
        return study_laterality([])
    for series_dir in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        try:
            audit = audit_series(series_dir, decode_sample_size=1)
        except FileNotFoundError:
            continue
        telemetry["header_read_failures"] += audit.header_read_failures
        evidence.append(
            SeriesLateralityEvidence(
                tag=audit.laterality_tag,
                geometry=audit.laterality_from_geometry,
                cross_tag_conflict=audit.laterality_cross_tag_conflict,
            )
        )
    return study_laterality(evidence)


def build_features_for(
    series_df: pd.DataFrame, series_root: Path, study_ids, timings=None
) -> np.ndarray:
    vectors = []
    for study_id in study_ids:
        study_start = time.perf_counter()
        study_dir = series_root / study_id
        # Total .dcm files in the study is the real I/O surface: ordering
        # validation reads every header of every candidate series, not just
        # the five slices ultimately decoded.
        study_slices = len(list(study_dir.rglob("*.dcm"))) if study_dir.is_dir() else 0
        planes = {}
        for plane in PLANES:
            candidates = validated_plane_candidates(series_df, series_root, study_id, plane)
            telemetry["candidates_tried"] += len(candidates)
            outcome = select_plane_sample([paths for _, paths in candidates])
            if outcome.absent or outcome.sample is None:
                telemetry["planes_absent"] += 1
                continue
            if outcome.candidates_tried > 1:
                telemetry["plane_retries"] += 1
            telemetry["decoded_slice_counts"].append(outcome.sample.decoded)
            winning_paths = candidates[outcome.candidates_tried - 1][1]
            header = pydicom.dcmread(winning_paths[0], stop_before_pixels=True)
            planes[plane] = PlaneInput(
                images=outcome.sample.images,
                image_orientation_patient=[float(v) for v in header.ImageOrientationPatient],
                pixel_spacing=[float(v) for v in header.PixelSpacing],
            )

        features = build_study_features(
            planes,
            _study_laterality(series_df, series_root, study_id),
            encode_batch,
            IMAGE_MEAN,
            IMAGE_STD,
        )
        if not features.laterality_reliable:
            telemetry["laterality_unreliable_studies"] += 1
        if not features.has_any_plane:
            telemetry["studies_with_no_plane"] += 1
        if timings is not None:
            timings.append(
                {
                    "slices": study_slices,
                    "seconds": time.perf_counter() - study_start,
                }
            )
        vectors.append(features.vector)
    return np.vstack(vectors)


import pydicom  # noqa: E402  (imported after the source path is established)

labeled_timings = []
extraction_start = time.perf_counter()
train_features = pd.DataFrame(
    build_features_for(
        train_series_df,
        DATA_DIR / "train_series",
        labeled_studies["StudyInstanceUID"],
        timings=labeled_timings,
    )
)
train_extraction_seconds = time.perf_counter() - extraction_start

In [ ]:
decoded_counts = pd.Series(telemetry["decoded_slice_counts"], dtype=float)

extraction_telemetry = pd.Series(
    {
        "Labeled studies": float(len(labeled_studies)),
        "Planes absent": float(telemetry["planes_absent"]),
        "Plane retries triggered": float(telemetry["plane_retries"]),
        "Candidates validated per plane (mean)": float(
            telemetry["candidates_tried"] / max(len(labeled_studies) * len(PLANES), 1)
        ),
        "Decoded slices per plane (mean)": (
            float(decoded_counts.mean()) if len(decoded_counts) else float("nan")
        ),
        "Decoded slices per plane (min)": (
            float(decoded_counts.min()) if len(decoded_counts) else float("nan")
        ),
        "Planes with fewer than five decoded": float((decoded_counts < SLICE_SAMPLE_SIZE).sum()),
        "Header read failures": float(telemetry["header_read_failures"]),
        "Studies with unreliable laterality": float(telemetry["laterality_unreliable_studies"]),
        "Studies with no usable plane": float(telemetry["studies_with_no_plane"]),
        "Feature extraction seconds": float(train_extraction_seconds),
    },
    name="Value",
).to_frame()

display(extraction_telemetry)

**Interpretation:** these counts are the only way several fallback paths become observable at all, because the sampled data has never exercised them. `Planes absent` and `Plane retries triggered` above zero would mean the same-plane retry and missing-plane fallback are doing real work rather than sitting unused. `Planes with fewer than five decoded` counts planes that fell back to a three- or four-slice mean. `Studies with unreliable laterality` is expected to be small but non-zero — roughly three of 58, predicted from the measured orientation distribution — and each such study is left untransformed with its flag set to 0 rather than flipped on a doubtful call. `Studies with no usable plane` must be **zero** for labeled training studies: a non-zero value is a data-path problem to diagnose, not a rounding detail.

## 4. Out-of-Fold Evaluation

In [ ]:
y = labeled_studies[LABEL_COLUMNS].astype(int).reset_index(drop=True)
selected_splits, folds = select_multilabel_folds(y, seed=SEED)

fold_identity = pd.Series(
    {
        "Labeled studies": len(labeled_studies),
        "Selected fold count": selected_splits,
        "Fold assignment signature": fold_signature(
            labeled_studies["StudyInstanceUID"].tolist(), folds
        ),
    },
    name="Value",
).to_frame()

display(fold_identity)

**Interpretation:** `select_multilabel_folds` is row-order-sensitive, so the same algorithm and seed do **not** by themselves imply the same fold membership as the report baseline. The signature digests the ordered study identifiers together with their fold assignment, so comparability between the two baselines is something this run demonstrates rather than assumes. The signature is a hash: it identifies an assignment without exposing which study went where.

In [ ]:
constant_predictions = pd.DataFrame(0.5, index=y.index, columns=LABEL_COLUMNS)
if macro_auc(y, constant_predictions) != 0.5:
    raise RuntimeError("Metric wiring check failed: a constant frame must score 0.5.")

cv_result = cross_validate_image_model(train_features, y, folds)

pooled_summary = pd.Series(
    {
        "Pooled OOF macro AUC": cv_result.pooled_macro_auc,
        "Fold macro AUC (mean)": float(np.mean(cv_result.fold_macro_auc)),
        "Fold macro AUC (min)": float(np.min(cv_result.fold_macro_auc)),
        "Fold macro AUC (max)": float(np.max(cv_result.fold_macro_auc)),
        "Constant-prediction sanity check": macro_auc(y, constant_predictions),
    },
    name="Value",
).to_frame()

display(pooled_summary)

**Interpretation:** `Pooled OOF macro AUC` is the primary metric and the only one that should be compared against the report baseline. The fold spread is **diagnostic only**: with 58 studies split across folds, individual fold scores are noisy, and a fold below 0.5 is not by itself evidence of a wiring error. The constant-prediction check is what would catch a genuine metric miswiring — it must be exactly 0.5, and the run aborts if it is not, because every score below it would otherwise be meaningless in a way no later inspection could detect.

In [ ]:
per_label_summary = (
    pd.Series(cv_result.pooled_per_label_auc, name="Pooled AUC")
    .reindex(LABEL_COLUMNS)
    .to_frame()
)
per_label_summary["Positives"] = [int(y[label].sum()) for label in LABEL_COLUMNS]

display(per_label_summary)

**Interpretation:** per-label AUC alongside the positive count that produced it, because a label with very few positives yields a score dominated by which side of a fold boundary those cases fell on. These are diagnostic: the pooled macro figure above is the contract's metric, and no per-label value here may be used to select a threshold, a feature, or a hyperparameter after the fact.

In [ ]:
flag_columns = train_features.iloc[:, CONTINUOUS_DIMENSIONS:]
flag_variance = pd.Series(
    {
        f"Flag {position} variance": float(flag_columns.iloc[:, position].var())
        for position in range(flag_columns.shape[1])
    },
    name="Value",
).to_frame()

display(flag_variance)

**Interpretation:** the four trailing flags are three plane-presence indicators and one laterality-reliability indicator. A variance of zero means the flag is constant across all 58 studies and therefore carries no information the head can learn from — its coefficient is unidentifiable from the intercept and is shrunk to approximately zero. This is the expected outcome given every audit measured complete plane coverage, and it is measured here rather than assumed. It does not indicate a fault: graceful degradation for a missing plane comes from excluding that plane from the mean, not from the flag.

## 5. Refit and Test Inference

In [ ]:
scaler, classifier = fit_image_model(train_features, y)

inference_start = time.perf_counter()
test_features = pd.DataFrame(
    build_features_for(test_series_df, DATA_DIR / "test_series", test_df["StudyInstanceUID"])
)
test_probabilities = classifier.predict_proba(scaler.transform(test_features.to_numpy()))
inference_seconds = time.perf_counter() - inference_start

In [ ]:
# The 58 labeled studies are not a representative timing sample: they are a
# fixed cohort, and per-study cost scales with how many DICOM files the study
# holds. Draw a supplemental sample stratified by that count, spanning the
# whole train corpus, so the projection rests on the range the hidden set
# will actually contain rather than on one cohort's middle.
TIMING_SAMPLE_PER_STRATUM = 5
TIMING_STRATA = 5

study_slice_totals = (
    train_series_df.groupby("StudyInstanceUID")["SeriesInstanceUID"].count().rename("series")
)
labeled_ids = set(labeled_studies["StudyInstanceUID"])
candidate_pool = study_slice_totals.loc[~study_slice_totals.index.isin(labeled_ids)]
strata = pd.qcut(candidate_pool.rank(method="first"), TIMING_STRATA, labels=False)

timing_rng = np.random.default_rng(SEED)
stratified_ids = []
for stratum in range(TIMING_STRATA):
    members = candidate_pool.index[strata == stratum].to_numpy()
    take = min(TIMING_SAMPLE_PER_STRATUM, len(members))
    stratified_ids.extend(timing_rng.choice(members, size=take, replace=False).tolist())

stratified_timings = []
build_features_for(
    train_series_df,
    DATA_DIR / "train_series",
    stratified_ids,
    timings=stratified_timings,
)

all_timings = pd.DataFrame(labeled_timings + stratified_timings)
all_timings["stratum"] = pd.qcut(
    all_timings["slices"].rank(method="first"), TIMING_STRATA, labels=False
)

timing_by_stratum = all_timings.groupby("stratum").agg(
    studies=("seconds", "count"),
    median_slices=("slices", "median"),
    mean_seconds=("seconds", "mean"),
    max_seconds=("seconds", "max"),
)

display(timing_by_stratum)

**Interpretation:** per-study cost scales with how many DICOM files a study holds, because ordering validation reads every header of every candidate series — not just the five slices ultimately decoded. Timing only the labeled cohort would measure one narrow band of that range, so this pools it with a supplemental sample stratified across the whole training corpus. A flat profile across strata would mean cost is dominated by fixed per-study work; a rising one means slice count is the driver, and the largest stratum is the one that determines whether a runtime budget holds.

In [ ]:
# The safety margin is not decoration. Decode cost was separately measured as
# I/O-contention-sensitive, rising roughly 2.7x when other work shared the
# kernel, and this projection extrapolates linearly from a sample far smaller
# than the hidden set. 3x covers the measured contention effect with room to
# spare; it is applied to the mean, and the slowest observed stratum is shown
# alongside so a pessimistic reading is available without recomputing.
RUNTIME_SAFETY_MARGIN = 3.0
DOCUMENTED_HIDDEN_STUDIES = 1300
RUNTIME_BUDGET_HOURS = 9.0

mean_seconds_per_study = float(all_timings["seconds"].mean())
slowest_stratum_seconds = float(timing_by_stratum["mean_seconds"].max())

timing_summary = pd.Series(
    {
        "Studies timed": float(len(all_timings)),
        "Slices per study (median)": float(all_timings["slices"].median()),
        "Slices per study (max)": float(all_timings["slices"].max()),
        "Seconds per study (mean)": mean_seconds_per_study,
        "Seconds per study (slowest stratum)": slowest_stratum_seconds,
        "Projected hours, mean rate": mean_seconds_per_study * DOCUMENTED_HIDDEN_STUDIES / 3600,
        "Projected hours, slowest stratum": (
            slowest_stratum_seconds * DOCUMENTED_HIDDEN_STUDIES / 3600
        ),
        "Safety margin applied": RUNTIME_SAFETY_MARGIN,
        "Projected hours with margin": (
            mean_seconds_per_study * DOCUMENTED_HIDDEN_STUDIES / 3600 * RUNTIME_SAFETY_MARGIN
        ),
        "Runtime budget (hours)": RUNTIME_BUDGET_HOURS,
        "Test inference seconds (visible studies)": float(inference_seconds),
    },
    name="Value",
).to_frame()

display(timing_summary)

**Interpretation:** this measures the **complete** path — series selection, ordering validation, decode, normalization, framing, canonicalization, encoding and the head — which is what a runtime budget actually turns on. An encoder-only figure understates it substantially, because decode and selection dominate. Two projections are given deliberately: the mean rate, and the slowest stratum's rate as a pessimistic bound. The margin covers the separately-measured I/O contention effect, where decode nearly tripled when other work shared the kernel; a figure measured on an idle kernel would otherwise flatter a busy one. Even the margin-bearing projection should be read as an order-of-magnitude check, since it extrapolates linearly to a set far larger than anything timed here.

In [ ]:
submission = build_submission(sample_df, test_df["StudyInstanceUID"], test_probabilities)
submission.to_csv("/kaggle/working/submission.csv", index=False)

submission_summary = pd.Series(
    {
        "Submission rows": float(len(submission)),
        "Submission columns": float(submission.shape[1]),
        "Probability minimum": float(test_probabilities.min()),
        "Probability maximum": float(test_probabilities.max()),
    },
    name="Value",
).to_frame()

display(submission_summary)

**Interpretation:** the submission is written once, inside the kernel, in the competition's own row and column order. Row and column counts confirm its shape without exposing any prediction, and the probability range confirms the head produced calibrated-looking output rather than saturating at 0 or 1. Writing this file is **not** the same as submitting it — that remains a separate and deliberate step.

## 6. Persisted Aggregate Summary

In [ ]:
summary = {
    "frozen_contract": json.loads(frozen_contract.to_json()),
    "environment": json.loads(environment_summary.to_json()),
    "extraction_telemetry": json.loads(extraction_telemetry.to_json()),
    "fold_identity": json.loads(fold_identity.to_json()),
    "pooled_scores": json.loads(pooled_summary.to_json()),
    "per_label_scores": json.loads(per_label_summary.to_json()),
    "flag_variance": json.loads(flag_variance.to_json()),
    "timing": json.loads(timing_summary.to_json()),
    "timing_by_stratum": json.loads(timing_by_stratum.to_json()),
    "submission_shape": json.loads(submission_summary.to_json()),
}

with open("/kaggle/working/image_baseline_summary.json", "w") as handle:
    json.dump(summary, handle, indent=2)

**Interpretation:** every aggregate above, gathered into one file so it can be retrieved from `/kaggle/working` after the run — Kaggle does not expose a notebook kernel's rendered output through its file API, only files written to the working directory and a plain log. Nothing in this file identifies a study or carries a per-study prediction.